# Super Mario Bros — Reinforcement Learning (Modernized 2026)

This notebook is an updated version of the original *Mario Tutorial* for **2026** libraries.  
It keeps the same teaching flow (Setup → Preprocess → Train → Test) but fixes all breaking changes so you can run it as-is.

### What changed vs the original version?
- **Python:** 3.9 → **3.13+** recommended (3.14 is Homebrew's current `python3` — fully supported) (3.10–3.12 works with constraints, see below). Original code broke on Python 3.9 + NumPy 2.
- **`gym` → `gymnasium`:** OpenAI Gym was unmaintained since 2022. All `import gym` / `from gym.wrappers ...` → `import gymnasium as gym` / `from gymnasium.wrappers ...`. See [Gymnasium migration guide](https://gymnasium.farama.org/introduction/migration_guide/).
- **`gym-super-mario-bros` 7.3.0 → 9.1.0** and **`nes-py` 8.2.1 → 9.0.1:** New releases target Gymnasium's `reset()`/`step()` semantics and fix the NumPy 2 `uint8` overflow (`Python integer 1024 out of bounds for uint8` in `nes_py/_rom.py:198`). See [commit history](https://github.com/Kautenja/gym-super-mario-bros).
- **`GrayScaleObservation` → `GrayscaleObservation`** (spelling) and moved from `gym.wrappers` to `gymnasium.wrappers`.
- **`env.reset()` / `env.step()` signatures:**
  - Old (Gym): `obs = env.reset()` and `obs, reward, done, info = env.step(a)`
  - New (Gymnasium): `obs, info = env.reset()` and `obs, reward, terminated, truncated, info = env.step(a)` with `done = terminated or truncated`
  - `VecEnv` (`DummyVecEnv`/`VecFrameStack`) still returns 4-tuple `obs, rewards, dones, infos` (it aggregates `terminated/truncated` internally).
- **`gym_super_mario_bros.make()` → `gymnasium.make()`:** `gym_super_mario_bros.make` is still an alias, but `gymnasium.make(..., render_mode='rgb_array')` is preferred. `render_mode="human"` now requires a display; for notebooks use `"rgb_array"` and plot with `matplotlib`.
- **`torch 1.10.1+cu113` → `torch 2.x`:** CUDA-specific wheel pins removed. `pip install torch` auto-selects CUDA/MPS/CPU. On macOS `torch` uses MPS if available.
- **`stable-baselines3[extra]`** already supports Gymnasium since SB3 2.0. Latest is **2.9.0** (Gymnasium 1.3.0).
- **Env wrapping bug fixed:** original used `DummyVecEnv([lambda: env])` sharing the same wrapped instance. Now uses a factory `lambda: make_mario_env()` that creates a fresh env per vector.
- **Training loop:** `while True:` infinite loop replaced with bounded `for` loop for safety in notebooks.

> **Teaching tip:** every updated code cell has a `# NOTE (2026):` comment explaining the change so you can compare with the original tutorial.


## 0. Environment Check
Run this first. If you see a warning you can still continue, but follow the fix it suggests.


In [ ]:
# NOTE (2026): Check Python / library versions before installing
# Original notebook assumed Python 3.9 + gym 0.26 + numpy 1.x. That combo now fails with:
#   OverflowError: Python integer 1024 out of bounds for uint8  (nes_py + numpy 2.0)
#   Gym has been unmaintained since 2022 ... please upgrade to Gymnasium
import sys, importlib.metadata as md
print(f"Python: {sys.version}")
try:
    print(f"numpy: {md.version('numpy')}")
except md.PackageNotFoundError:
    print("numpy: not installed yet")
try:
    print(f"gymnasium: {md.version('gymnasium')}")
except md.PackageNotFoundError:
    print("gymnasium: not installed yet (will be installed next cell)")
try:
    print(f"gym-super-mario-bros: {md.version('gym-super-mario-bros')}")
except md.PackageNotFoundError:
    print("gym-super-mario-bros: not installed yet")
try:
    print(f"nes-py: {md.version('nes-py')}")
except md.PackageNotFoundError:
    print("nes-py: not installed yet")
try:
    print(f"stable-baselines3: {md.version('stable-baselines3')}")
    print(f"torch: {md.version('torch')}")
except md.PackageNotFoundError:
    print("SB3/torch: not installed yet")

# Python version guidance
if sys.version_info < (3, 10):
    print("\n⚠️ Python <3.10 detected — newest gym-super-mario-bros 9.x requires Python >=3.13.")
    print("   Option A (recommended): create a fresh env with Python 3.13:  python3.14 -m venv .venv (or python3.13) && source .venv/bin/activate")
    print("   Option B: pin older deps: pip install 'numpy<2' 'gym==0.26.2' 'gym-super-mario-bros==7.4.0' 'nes-py==8.2.1'")
elif sys.version_info < (3, 13):
    print("\nℹ️ Python 3.10-3.12 detected — you can run the modern stack, but pip may cap gym-super-mario-bros to 7.4.0.")
    print("   For the very latest 9.x (tested 2026-06) upgrade to Python 3.13: brew install python@3.13  (or python@3.14 — now default)")
else:
    print("\n✅ Python >=3.13 (3.14 works too) — you can install the newest gym-super-mario-bros 9.1.0 + nes-py 9.0.1 + gymnasium 1.3.0")


# 1. Setup Mario

In [ ]:
# NOTE (2026): Single install cell for the modern stack
# Old: !pip install gym_super_mario_bros==7.3.0 nes_py   (pinned, now broken with numpy 2.0)
# New: gymnasium replaces gym, gym-super-mario-bros>=9 + nes-py>=9 fix numpy 2 overflow
#      SB3 2.9 + torch 2.x are gymnasium-native. No CUDA-specific wheel pin needed.
# Run this cell once per environment. Re-running is safe.
%pip install --upgrade pip wheel setuptools
%pip install gymnasium gym-super-mario-bros nes-py "stable-baselines3[extra]" torch matplotlib tqdm tensorboard rich
# If you need a specific Python constraint:
#   Python >=3.13 (incl. 3.14): installs gym-super-mario-bros 9.1.0, nes-py 9.0.1, gymnasium 1.3.0, torch 2.14, SB3 2.9 (recommended)
#   Python 3.10-3.12: pip will auto-pick gym-super-mario-bros 7.4.0 / nes-py 8.2.1 — still works but set 'numpy<2' if you hit uint8 overflow
#   Python 3.9: prefer to upgrade; fallback: %pip install "numpy<2" after the line above


In [ ]:
# NOTE (2026): Imports updated for Gymnasium
# Old: import gym_super_mario_bros ; from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
#      from nes_py.wrappers import JoypadSpace  (unchanged)
# New: import gymnasium as gym  (Gym is now Gymnasium, drop-in replacement)
#      JoypadSpace still from nes_py.wrappers — it now wraps a Gymnasium env and returns 5-tuple.
# Import order matters: import gym_super_mario_bros registers envs with gymnasium.
import gymnasium as gym  # NOTE (2026): was `import gym` — gymnasium is the maintained fork
import gym_super_mario_bros  # registers SuperMarioBros-* envs
from nes_py.wrappers import JoypadSpace
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT, RIGHT_ONLY, COMPLEX_MOVEMENT

print("gymnasium version:", gym.__version__)
print("SIMPLE_MOVEMENT:", SIMPLE_MOVEMENT)
print("RIGHT_ONLY:", RIGHT_ONLY)
# Verify registration
print("Available Mario envs:", [s for s in gym.envs.registry.keys() if "SuperMario" in s][:5], "...")


In [ ]:
# NOTE (2026): Environment creation now requires render_mode
# Old: env = gym_super_mario_bros.make('SuperMarioBros-v3')  # gym alias, no render_mode, v3 deprecated
# New: env = gym.make('SuperMarioBros-v0', render_mode='rgb_array')  # gymnasium.make, explicit render_mode
#      v0=vanilla ROM, v1=downsample, v2=pixel, v3=rectangle — v0 is default and most compatible
#      Use render_mode='human' only if you have a window/display; notebooks should use 'rgb_array'
# JoypadSpace still maps 256 NES buttons -> discrete SIMPLE_MOVEMENT (7 actions)
env = gym.make('SuperMarioBros-v0', render_mode='rgb_array')
env = JoypadSpace(env, SIMPLE_MOVEMENT)

print("Action space:", env.action_space, "->", env.get_action_meanings())
print("Observation space:", env.observation_space)  # Box(0,255,(240,256,3), uint8)
print("Env created successfully — observation shape:", env.observation_space.shape)
print("\n# TIP: try other action sets: RIGHT_ONLY (2 actions, easier) or COMPLEX_MOVEMENT (12 actions)")


In [ ]:
# NOTE (2026): reset() / step() API changed
# Old (Gym): obs = env.reset() ; obs, reward, done, info = env.step(a) ; done is single bool
# New (Gymnasium): obs, info = env.reset() ; obs, reward, terminated, truncated, info = env.step(a)
#   terminated = game death / flag, truncated = time limit (9999999 wrapper). Use done = terminated or truncated
#   Call env.reset(seed=123) for reproducibility.
import matplotlib.pyplot as plt

obs, info = env.reset(seed=42)  # NOTE (2026): reset now returns (obs, info) tuple
print(f"Initial obs shape: {obs.shape}, info keys: {list(info.keys())[:6]}...")
print(f"Reward components available:", info.get('reward_components', 'N/A'))

done = False
total_reward = 0
for step in range(1000):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)  # NOTE (2026): 5-tuple
    done = terminated or truncated  # NOTE (2026): combine both flags
    total_reward += reward
    if done:
        print(f"Episode done at step {step} (terminated={terminated}, truncated={truncated}), total_reward={total_reward:.1f}")
        obs, info = env.reset()  # reset after done
        total_reward = 0
    if step == 5:
        print(f"Step 5: reward={reward:.2f}, x_pos={info.get('x_pos')}, coins={info.get('coins')}")
        # Bonus: render one frame with matplotlib (replaces env.render() which needs a window)
        plt.figure(figsize=(4,4))
        plt.imshow(obs)
        plt.title(f"Random agent frame at step {step}")
        plt.axis('off')
        plt.show()
        break  # short demo — remove break to run full 1000 steps

print(f"\nRandom rollout test passed. Total reward after demo: {total_reward:.2f}")
env.close()


# 2. Preprocess Environment
We convert to grayscale, vectorize, and stack frames so the CNN sees motion.

**Why?** Raw frames are `(240,256,3)` RGB. Grayscale → 1 channel, FrameStack 4 → `(240,256,4)` lets PPO see velocity. This matches the SB3 Atari pattern.


In [ ]:
# NOTE (2026): No separate torch/SB3 pin needed — already installed above.
# Old cells:
#   !pip install torch==1.10.1+cu113 torchvision==0.11.2+cu113 ...  (CUDA 11.3, now obsolete)
#   !pip install stable-baselines3[extra]
# New: single line above does `pip install torch stable-baselines3`. On Apple Silicon torch uses MPS automatically.
import torch
print(f"torch {torch.__version__} — cuda available: {torch.cuda.is_available()}, mps available: {torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False}")
import stable_baselines3
print(f"stable-baselines3 {stable_baselines3.__version__}")
# Optional: check if tensorboard is available for PPO logging
try:
    import tensorboard
    print("tensorboard available for PPO logging")
except ImportError:
    print("tensorboard not installed — PPO still works, logs just won't show in TensorBoard (pip install tensorboard to enable)")


In [ ]:
# NOTE (2026): Wrapper imports moved
# Old: from gym.wrappers import GrayScaleObservation  (misspelling + gym)
# New: from gymnasium.wrappers import GrayscaleObservation  (correct spelling + gymnasium)
# Also: from stable_baselines3.common.vec_env import VecFrameStack, DummyVecEnv (unchanged)
# Optional extra: gymnasium.wrappers also has ResizeObservation, FrameStackObservation if you want 84x84.
from gymnasium.wrappers import GrayscaleObservation  # NOTE: was gym.wrappers.GrayScaleObservation
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from matplotlib import pyplot as plt
print("Wrappers imported successfully")
print("GrayscaleObservation doc:", GrayscaleObservation.__doc__[:120], "...")


In [ ]:
# NOTE (2026): Fixed DummyVecEnv factory + added render_mode + Grayscale spelling
# Old (buggy):
#   env = gym_super_mario_bros.make('SuperMarioBros-v0')
#   env = JoypadSpace(env, SIMPLE_MOVEMENT)
#   env = GrayScaleObservation(env, keep_dim=True)  # typo & old gym
#   env = DummyVecEnv([lambda: env])  # reuses same env instance — breaks vectorization subtly
#   env = VecFrameStack(env, 4, channels_order='last')
# New: factory creates a FRESH env each call; gymnasium.make with render_mode; GrayscaleObservation
def make_mario_env():
    # 1. Create base env with explicit render_mode
    env = gym.make('SuperMarioBros-v0', render_mode='rgb_array')
    # 2. Simplify controls (7 discrete actions)
    env = JoypadSpace(env, SIMPLE_MOVEMENT)
    # 3. Grayscale — keep_dim=True keeps shape (240,256,1) so VecFrameStack yields (240,256,4)
    env = GrayscaleObservation(env, keep_dim=True)
    return env

# 4. Vectorize — DummyVecEnv runs 1 env in same process (use SubprocVecEnv for >1)
vec_env = DummyVecEnv([make_mario_env])  # NOTE: pass factory, not pre-made env
# 5. Stack 4 frames — channels_order='last' matches PyTorch CNN (H,W,C*N)
vec_env = VecFrameStack(vec_env, n_stack=4, channels_order='last')

print("Vectorized env:", vec_env)
print("Observation space:", vec_env.observation_space)  # Box(0,255,(240,256,4))
print("Action space:", vec_env.action_space)


In [ ]:
obs = vec_env.reset()  # VecEnv reset still returns just obs (no info tuple) — SB3 handles it
print("After VecFrameStack, obs shape:", obs.shape)  # (1, 240, 256, 4) — batch=1, H=240, W=256, stacked=4
print("obs dtype:", obs.dtype, "range:", obs.min(), "-", obs.max())
# NOTE: First reset duplicates initial frame 4x (SB3 VecFrameStack behavior). After steps you'll see motion.


In [ ]:
# NOTE (2026): VecEnv step returns 4-tuple, not Gymnasium's 5-tuple
# VecEnv abstracts gymnasium's terminated/truncated into single `dones` array for SB3.
# Old: state, reward, done, info = env.step([5])  # worked with gym VecEnv
# New: still obs, rewards, dones, infos = vec_env.step([5])  — same signature, now wraps gymnasium under the hood
obs, rewards, dones, infos = vec_env.step([5])  # action 5 = e.g., 'right + A' in SIMPLE_MOVEMENT
print("obs shape after step:", obs.shape)
print("rewards:", rewards, "dones:", dones)
print("infos[0] keys sample:", list(infos[0].keys())[:6])


In [ ]:
# Visualize the 4 stacked grayscale frames — shows motion history
plt.figure(figsize=(16, 4))
for idx in range(obs.shape[3]):  # 4 stacked frames, channels_last
    plt.subplot(1, 4, idx+1)
    plt.imshow(obs[0, :, :, idx], cmap='gray')
    plt.title(f"Stacked frame {idx+1}")
    plt.axis('off')
plt.suptitle("VecFrameStack (4 grayscale frames) — note motion across frames after step")
plt.tight_layout()
plt.show()
# TEACHING NOTE: If frames look identical, that's expected on reset. Step a few times to see Mario move.


In [ ]:
# Bonus: show that stepping a few times creates motion (optional demo)
for _ in range(10):
    obs, rewards, dones, infos = vec_env.step([vec_env.action_space.sample()])
plt.figure(figsize=(16,4))
for idx in range(obs.shape[3]):
    plt.subplot(1,4,idx+1)
    plt.imshow(obs[0,:,:,idx], cmap='gray')
    plt.axis('off')
plt.suptitle("After 10 random steps — you can see Mario's x_pos progress in stacked frames")
plt.show()
print(f"Current x_pos: {infos[0].get('x_pos')}, reward: {rewards[0]:.2f}")


# 3. Train the RL Model
We use **PPO** (Proximal Policy Optimization) with a CNN policy — same as the original, now on latest SB3 2.9.


In [ ]:
# NOTE (2026): Imports unchanged, but now gymnasium-native SB3
import os
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
print(f"PPO available, SB3 version: {stable_baselines3.__version__}")


In [ ]:
class TrainAndLoggingCallback(BaseCallback):
    """Save model every `check_freq` steps — same as original, comments added."""
    def __init__(self, check_freq, save_path, verbose=1):
        super().__init__(verbose)
        self.check_freq = check_freq
        self.save_path = save_path

    def _init_callback(self):
        if self.save_path is not None:
            os.makedirs(self.save_path, exist_ok=True)

    def _on_step(self) -> bool:
        if self.n_calls % self.check_freq == 0:
            model_path = os.path.join(self.save_path, f'best_model_{self.n_calls}')
            self.model.save(model_path)
            if self.verbose:
                print(f"Saved checkpoint to {model_path}")
        return True

# NOTE (2026): No change — callback still works with Gymnasium. SB3's BaseCallback is version-agnostic.


In [ ]:
CHECKPOINT_DIR = './train/'
LOG_DIR = './logs/'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
print(f"Checkpoints -> {os.path.abspath(CHECKPOINT_DIR)}")
print(f"TensorBoard logs -> {os.path.abspath(LOG_DIR)}")
print("View with: tensorboard --logdir ./logs/")


In [ ]:
# NOTE (2026): PPO setup — learning_rate and n_steps kept as original for reproducibility
# Original used learning_rate=0.000001 (1e-6) and n_steps=512. SB3 default is 3e-4 — we keep 1e-6 as per tutorial.
# Add device="auto" to use MPS on Mac if available (torch 2.x handles it).
callback = TrainAndLoggingCallback(check_freq=10000, save_path=CHECKPOINT_DIR)
print("Callback ready — saves every 10k steps")


In [ ]:
# NOTE (2026): CnnPolicy now uses gymnasium spaces under the hood — no code change needed
# vec_env already is DummyVecEnv(VecFrameStack(Grayscale(...))) so shape is (240,256,4)
# If tensorboard is not installed, disable logging gracefully
try:
    import tensorboard  # noqa: F401
    tb_log = LOG_DIR
except ImportError:
    tb_log = None
    print("tensorboard not found — training without TensorBoard logging (pip install tensorboard to enable)")
model = PPO(
    'CnnPolicy',
    vec_env,
    verbose=1,
    tensorboard_log=tb_log,
    learning_rate=1e-6,  # NOTE: original tutorial's tiny LR — consider 3e-4 for faster learning
    n_steps=512,         # rollout buffer size
    # device="auto",    # uncomment to force cpu/cuda/mps if needed
)
print("PPO model created")
print(model.policy)


In [ ]:
# NOTE (2026): Training — same call, but prefer smaller timesteps for notebook demo
# Original: model.learn(total_timesteps=1000000, callback=callback)  # 1M steps ~ hours
# For a quick demo we do 20k steps. Increase to 1M when you want a real agent.
# Remove the callback for very short runs, or keep it to test checkpoint saving.
# This cell will take ~30-60s for 20k steps on M1/M2/M3 Mac; 1M steps takes longer.
DEMO_STEPS = 20000  # change to 1000000 for full training
try:
    model.learn(total_timesteps=DEMO_STEPS, callback=callback, progress_bar=True)
except ImportError as e:
    # Fallback if rich/tensorboard not installed (e.g., stable-baselines3 without [extra])
    print(f"Progress bar / tensorboard not available ({e}), retrying without progress_bar")
    model.learn(total_timesteps=DEMO_STEPS, callback=callback, progress_bar=False)
print(f"Training completed for {DEMO_STEPS} steps")


In [ ]:
model.save('thisisatestmodel')
print("Saved to thisisatestmodel.zip")
print("Checkpoint models also in", CHECKPOINT_DIR, ":", os.listdir(CHECKPOINT_DIR)[:5])


# 4. Test it Out
Load a saved model and watch it play. Note the environment is still the vectorized one.


In [ ]:
# NOTE (2026): PPO.load works with gymnasium SB3 models
# Old: model = PPO.load('./train/best_model_1000000')  # assumed 1M checkpoint
# New: load whatever exists — demo used 20k, so we try best_model_20000 then fallback to thisisatestmodel
import glob
candidates = glob.glob('./train/best_model_*') + ['thisisatestmodel.zip', 'thisisatestmodel']
print("Candidates:", candidates[:5])
load_path = None
for c in ['./train/best_model_20000', './train/best_model_20000.zip', 'thisisatestmodel', 'thisisatestmodel.zip']:
    if os.path.exists(c) or os.path.exists(c+'.zip'):
        load_path = c.replace('.zip','')
        break
if load_path:
    model = PPO.load(load_path)
    print(f"Loaded model from {load_path}")
else:
    print("No saved model found — using current in-memory model from training")
    # model already in memory from previous cell


In [ ]:
# For testing we can reuse vec_env, or create a fresh one for rendering.
# VecEnv reset: obs = vec_env.reset() (no info tuple, unlike raw gymnasium env)
obs = vec_env.reset()
print("Reset obs shape:", obs.shape)
print("Environment ready for inference")


In [ ]:
# NOTE (2026): Inference loop updated
# Old:
#   state = env.reset()
#   while True:
#       action, _ = model.predict(state)
#       state, reward, done, info = env.step(action)  # 4-tuple + infinite loop
#       env.render()  # human window — blocks in notebook
# New:
#   VecEnv uses 4-tuple (obs, rewards, dones, infos). For raw env use 5-tuple.
#   We bound the loop (1000 steps) and plot with matplotlib instead of env.render()
import time
obs = vec_env.reset()
total_reward = 0
frames = []  # collect a few frames to display later

for step in range(500):  # bounded for notebook safety — original had `while True`
    action, _states = model.predict(obs, deterministic=True)  # deterministic for evaluation
    obs, rewards, dones, infos = vec_env.step(action)  # VecEnv 4-tuple
    total_reward += rewards[0]
    # Save frame for visualization (every 50 steps)
    if step % 50 == 0:
        # obs is (1,240,256,4) — show the latest stacked channel
        frames.append(obs[0, :, :, -1].copy())
    if dones[0]:
        print(f"Episode done at step {step}, total_reward={total_reward:.1f}, x_pos={infos[0].get('x_pos')}")
        obs = vec_env.reset()
        total_reward = 0
    if step % 100 == 0:
        print(f"Step {step}: action={action[0]}, reward={rewards[0]:.2f}, x_pos={infos[0].get('x_pos')}")

print(f"\nInference rollout finished")
if frames:
    plt.figure(figsize=(12,3))
    for i, f in enumerate(frames[:4]):
        plt.subplot(1,4,i+1)
        plt.imshow(f, cmap='gray')
        plt.title(f"Inference frame {i*50}")
        plt.axis('off')
    plt.show()


In [ ]:
# Optional: evaluate with raw (non-vectorized) env to see RGB frames and info dict
raw_env = gym.make('SuperMarioBros-v0', render_mode='rgb_array')
raw_env = JoypadSpace(raw_env, SIMPLE_MOVEMENT)
obs, info = raw_env.reset()
for _ in range(3):
    action, _ = model.predict(vec_env.reset(), deterministic=True)  # predict needs VecEnv obs shape; demo only
    # For raw env we need to adapt: convert grayscale stacked obs -> not directly compatible, so we just do random here
    obs, reward, terminated, truncated, info = raw_env.step(raw_env.action_space.sample())
    print(f"Raw env step: reward={reward:.2f}, x_pos={info['x_pos']}, terminated={terminated}")
    plt.imshow(obs)
    plt.title(f"Raw RGB frame — x_pos {info['x_pos']}")
    plt.axis('off')
    plt.show()
    if terminated or truncated:
        obs, info = raw_env.reset()
raw_env.close()
vec_env.close()
print("\nEnvs closed. ✅ Tutorial complete!")
print("Next steps: train longer (1M steps), try COMPLEX_MOVEMENT, or log to TensorBoard: tensorboard --logdir ./logs/")


In [ ]:
# Cleanup (optional)
# vec_env.close() already called above. If you re-run the notebook, just re-execute from the Setup cells.
print("Done. To retrain, re-run the Train cells.")
